# V1 prompt-only paired pilot
4 new public tasks, 8 runs; no audited task, reference answer, or audit notes are mounted.
A completed execution is not a passing repair.

## Controller and metrics

In [ ]:
"""Notebook support code, embedded for Kaggle; no agent tools or model replacement."""
import asyncio
from datetime import datetime, timezone
import hashlib
import json
import logging
from pathlib import Path
import shutil
import stat
import time
import traceback
import zipfile

MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
SUBMISSION_FILES = {'agent.yaml', 'eval_config.yaml', 'prompts/system.md'}


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(data, indent=2, default=str) + '\n', encoding='utf-8')
    tmp.replace(path)


def one_path(candidates, label):
    candidates = sorted({Path(p).resolve() for p in candidates})
    if len(candidates) != 1:
        raise RuntimeError(f'{label}: expected one mounted input, found {candidates}. '
                           'Attach the official input; remove duplicate mounts or set an explicit path.')
    return candidates[0]


def discover_inputs(root, overrides=None):
    root = Path(root)
    overrides = overrides or {}
    # File signatures support both old and new Kaggle mount layouts.
    data = Path(overrides['data']) if overrides.get('data') else one_path(
        [p.parent for p in root.rglob('tasks.jsonl') if (p.parent / 'snapshots').is_dir()],
        'Competition data (401 means account authorization; no downloads attempted)')
    wheels = Path(overrides['wheelhouse']) if overrides.get('wheelhouse') else one_path(
        [p.parent for p in root.rglob('swegemma-*.whl')], 'Official wheelhouse')
    model = Path(overrides['model']) if overrides.get('model') else one_path(
        [p.parent for p in root.rglob('config.json')
         if MODEL_NAME in p.parts and any(p.parent.glob('*.safetensors'))], '31B model')
    archive = Path(overrides['zip']) if overrides.get('zip') else one_path(
        [p for name in ('submission.zip', 'submission.bundle') for p in root.rglob(name)
         if not any(p.is_relative_to(excluded) for excluded in (data, wheels, model))], 'V0 submission.zip')
    for path in (data, wheels, model, archive):
        if not path.exists() or not path.resolve().is_relative_to(root.resolve()):
            raise RuntimeError(f'Input is absent or outside Kaggle input root: {path}')
    for name in ('tasks.jsonl', 'snapshots', 'wheels', 'graphs', 'embeddings',
                 'HARNESS_README.md', 'sample_submission/agent.yaml'):
        if not (data / name).exists():
            raise RuntimeError(f'Missing official competition input: {data / name}')
    if MODEL_NAME not in model.parts or not (model / 'config.json').is_file() or not any(model.glob('*.safetensors')):
        raise RuntimeError('Expected mounted official 31B model config and safetensors weights')
    return {'data': data, 'wheelhouse': wheels, 'model': model, 'zip': archive}


def extract_submission(archive_path, destination):
    destination = Path(destination)
    if destination.exists():
        raise ValueError('Use a fresh run directory; never overwrite a previous submission')
    with zipfile.ZipFile(archive_path) as archive:
        infos = archive.infolist()
        if len(infos) != len(SUBMISSION_FILES) or {i.filename for i in infos} != SUBMISSION_FILES:
            raise ValueError('V0 ZIP must contain exactly agent.yaml, eval_config.yaml, prompts/system.md')
        if sum(i.file_size for i in infos) > 1024 * 1024:
            raise ValueError('V0 ZIP exceeds the 1 MiB local preflight limit')
        if any(stat.S_ISLNK(i.external_attr >> 16) or i.flag_bits & 1 for i in infos):
            raise ValueError('Symlink or encrypted ZIP entry is unsupported')
        if archive.testzip() is not None:
            raise ValueError('ZIP CRC verification failed')
        archive.extractall(destination)
    return hashlib.sha256(Path(archive_path).read_bytes()).hexdigest()


def budget_profile(mode):
    if mode == 'single':
        return dict(max_time_minutes=60.0, max_tool_calls=100, timeout_seconds=300,
                    verification_timeout_seconds=300, max_tasks=1, task_wall_minutes=75.0)
    if mode == 'batch':
        # 50 * 12 + 45 bootstrap + 15 finalization = 660 minutes < 720.
        return dict(max_time_minutes=8.0, max_tool_calls=60, timeout_seconds=120,
                    verification_timeout_seconds=300, max_tasks=50, task_wall_minutes=12.0)
    raise ValueError('RUN_MODE must be single or batch')


def admission_seconds(session_started, task_wall_minutes, now=None):
    # A 705-minute soft deadline leaves 15 minutes before the 12-hour ceiling.
    now = time.monotonic() if now is None else now
    remaining = 705 * 60 - (now - session_started)
    return task_wall_minutes * 60 if remaining >= task_wall_minutes * 60 else 0.0


def preserve_server_log(server, output):
    path = getattr(server, 'log_path', None)
    if path and Path(path).is_file():
        shutil.copyfile(path, Path(output) / 'vllm.log')


def exception_record(output, stage, exc):
    write_json(Path(output) / 'exception.json', dict(
        stage=stage, exception_type=type(exc).__name__, message=str(exc),
        traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)),
        recorded_at=datetime.now(timezone.utc).isoformat(),
        real_task_completed=False, repair_score=None))


def trace_from_result(result):
    from adk_eval_core.tracing import SessionTrace
    trace = result.trace
    if trace is None and getattr(result, 'trace_path', None):
        path = Path(result.trace_path)
        if path.is_file():
            trace = SessionTrace.from_json(path)
    return trace


def persist_result(result, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    # Preserve even empty patches and outputs: absence is a result, not a pass.
    (directory / 'agent.patch').write_text(result.agent_patch or '', encoding='utf-8')
    (directory / 'phase2.log').write_text(result.test_output or '', encoding='utf-8')
    trace = trace_from_result(result)
    entries = []
    if trace is not None:
        trace.save(directory / 'trace.json')
        trace.save(directory / 'trace_legacy.json', format='legacy')
        entries = trace.to_dict(format='legacy')['entries']
    tool_events = [e for e in entries if e['type'] in ('tool_call', 'tool_response')]
    model_events = [e for e in entries if e['type'] not in ('tool_call', 'tool_response') or e.get('usage')]
    for name, events in [('tool_events.jsonl', tool_events), ('model_events.jsonl', model_events)]:
        (directory / name).write_text(''.join(json.dumps(e) + '\n' for e in events), encoding='utf-8')
    submitted = False
    for event in tool_events:
        if event['type'] == 'tool_response' and event.get('tool') == 'submit_patch':
            try:
                submitted |= json.loads(event.get('result', '{}')).get('status') == 'ok'
            except (ValueError, AttributeError):
                pass
    # verify_task's final test result contains this exact stdout/stderr envelope.
    # Patch/setup errors can also have a nonzero code, so code alone is insufficient.
    phase2_ran = (result.test_exit_code >= 0 and (result.test_output or '').startswith('STDOUT:\n')
                  and '\nSTDERR:\n' in result.test_output)
    has_inference = result.total_llm_calls > 0 and any(e.get('usage') or e['type'] in
                   ('thinking', 'text', 'final', 'tool_call') for e in entries)
    record = dict(instance_id=result.instance_id, repo=result.repo, resolved=result.resolved,
                  error=result.error, test_exit_code=result.test_exit_code,
                  phase2_tests_ran=phase2_ran, total_llm_calls=result.total_llm_calls,
                  tool_calls=result.tool_calls, duration_seconds=result.duration_seconds,
                  patch_chars=len(result.agent_patch or ''), explicit_submit_patch_success=submitted,
                  model_events_present=has_inference,
                  first_task_goal_completed=bool(has_inference and submitted and result.agent_patch
                                                and phase2_ran),
                  # Retain official result; never turn missing evidence into a score.
                  competition_score=None)
    write_json(directory / 'result.json', record)
    return record


async def evaluate_with_deadline(evaluator, task, context, directory, deadline_seconds, server=None):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    pending = asyncio.create_task(evaluator.evaluate_task(task=task, task_index=1,
                                  total_tasks=1, context=context))
    try:
        while True:
            remaining = deadline_seconds - (time.monotonic() - started)
            write_json(directory / 'live_status.json', dict(
                instance_id=task.instance_id, elapsed_seconds=time.monotonic() - started,
                phase=str(getattr(context, 'current_phase', 'unknown')),
                total_llm_calls=context.llm_calls_used, tool_calls=context.tool_calls_used,
                patch_submitted=context.patch_submitted, completed=False))
            if context.submitted_patch is not None:
                (directory / 'last_submitted.patch').write_text(context.submitted_patch, encoding='utf-8')
            if server is not None:
                preserve_server_log(server, directory.parent)
            if remaining <= 0:
                raise TimeoutError(f'Outer task wall-clock budget exhausted ({deadline_seconds}s)')
            done, _ = await asyncio.wait({pending}, timeout=min(15.0, remaining))
            if done:
                result = pending.result()
                record = persist_result(result, directory)
                write_json(directory / 'live_status.json', dict(
                    instance_id=task.instance_id, completed=True, result=record))
                return record
    except BaseException as exc:
        exception_record(directory, 'evaluate_task', exc)
        if context.submitted_patch is not None:
            (directory / 'last_submitted.patch').write_text(context.submitted_patch, encoding='utf-8')
        if not pending.done():
            pending.cancel()
            # Cancellation of underlying synchronous work may take longer; do not
            # wait indefinitely for cleanup and do not claim a completed Phase 2.
            await asyncio.wait({pending}, timeout=30)
        # Retrieve completed exceptions to avoid "Task exception was never retrieved".
        if pending.done() and not pending.cancelled():
            pending.exception()
        raise


def bundle_run(run_dir):
    run_dir = Path(run_dir)
    return Path(shutil.make_archive(str(run_dir), 'zip', root_dir=run_dir))

"""Conservative metrics; missing/incomplete runs never become passing scores."""
import re
from statistics import mean


def classify(result, output, patch):
    invoked = output.startswith('STDOUT:\n') and '\nSTDERR:\n' in output
    collection = invoked and bool(re.search(r'ERROR collecting|errors? during collection', output, re.I))
    # This is an application-validity metric, not a semantic correctness claim.
    valid = bool(patch and invoked and 'Failed to apply agent patch' not in output)
    return dict(phase2_invoked=invoked, collection_failed=collection,
                patch_application_valid=valid,
                test_bodies_observed=bool(invoked and re.search(r'\b\d+ (?:passed|failed)\b', output)),
                verifier_timeout=result.get('test_exit_code') == 124)


def summarize(records, schedule):
    output = {'planned_runs': len(schedule), 'observed_runs': len(records),
              'complete': len(records) == len(schedule), 'competition_score': None, 'arms': {}}
    for arm in ('v0', 'v1'):
        planned = [r for r in schedule if r['arm'] == arm]
        actual = [r for r in records if r['arm'] == arm]
        invoked = [r for r in actual if r.get('phase2_invoked')]
        observed = [r for r in actual if r.get('duration_seconds') is not None]
        n = len(planned)
        output['arms'][arm] = dict(planned=n, observed=len(actual),
            resolved=sum(r.get('resolved') is True for r in actual),
            resolution_rate=sum(r.get('resolved') is True for r in actual) / n if n and len(actual) == n else None,
            patch_validity_rate=sum(r.get('patch_application_valid') is True for r in actual) / n if n and len(actual) == n else None,
            collection_failure_rate_all_planned=sum(r.get('collection_failed') is True for r in actual) / n if n and len(actual) == n else None,
            collection_failure_rate_invoked=sum(r.get('collection_failed') is True for r in invoked) / len(invoked) if invoked else None,
            phase2_invoked=len(invoked), timeouts=sum(r.get('outer_timeout') is True or r.get('verifier_timeout') is True or r.get('agent_timeout') is True for r in actual),
            errors=sum(bool(r.get('execution_error')) for r in actual),
            mean_llm_calls=mean(r['total_llm_calls'] for r in actual if r.get('total_llm_calls') is not None) if any(r.get('total_llm_calls') is not None for r in actual) else None,
            mean_tool_calls=mean(r['tool_calls'] for r in actual if r.get('tool_calls') is not None) if any(r.get('tool_calls') is not None for r in actual) else None,
            mean_duration_seconds=mean(r['duration_seconds'] for r in observed) if observed else None)
    pairs = {}
    for row in records:
        pairs.setdefault(row['instance_id'], {})[row['arm']] = row
    paired = [v for v in pairs.values() if set(v) == {'v0', 'v1'}]
    output['paired'] = dict(n=len(paired),
        v1_only_resolved=sum(p['v1'].get('resolved') is True and p['v0'].get('resolved') is not True for p in paired),
        v0_only_resolved=sum(p['v0'].get('resolved') is True and p['v1'].get('resolved') is not True for p in paired))
    return output

"""Embedded comparison controller. Not an Agent tool or a solution provider."""
import re
import hashlib
import json
import stat
import zipfile


def unpack_experiment(archive, destination):
    expected = {'experiment.json'} | {f'{a}/{n}' for a in ('v0', 'v1')
        for n in ('agent.yaml', 'eval_config.yaml', 'prompts/system.md')}
    with zipfile.ZipFile(archive) as z:
        infos = z.infolist()
        if {i.filename for i in infos} != expected or len(infos) != len(expected):
            raise ValueError('Experiment payload must contain only the two agents and frozen plan')
        if sum(i.file_size for i in infos) > 1024 * 1024:
            raise ValueError('Experiment payload too large')
        if any(stat.S_ISLNK(i.external_attr >> 16) or i.flag_bits & 1 for i in infos):
            raise ValueError('Symlink/encrypted payload rejected')
        if z.testzip() is not None:
            raise ValueError('Payload CRC failed')
        if destination.exists():
            raise FileExistsError('Use a fresh evidence directory')
        z.extractall(destination)
    plan = json.loads((destination / 'experiment.json').read_text())
    if plan['historical_task_excluded'] != 'fastapi_15661' or any(
            r['instance_id'] == plan['historical_task_excluded'] for r in plan['schedule']):
        raise ValueError('Audited task cannot enter generalization pilot')
    for name, digest in plan['source_hashes'].items():
        # Do not trust plan paths, even though archive entry names were checked.
        if name not in expected - {'experiment.json'}:
            raise ValueError('Invalid source hash path')
        if hashlib.sha256((destination / name).read_bytes()).hexdigest() != digest:
            raise ValueError(f'Frozen source mismatch: {name}')
    if set(plan['source_hashes']) != expected - {'experiment.json'}:
        raise ValueError('Missing source hash')
    return plan


async def run_comparison():
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    records = []
    failure = None
    try:
        task_map = {t.instance_id: t for t in selected}
        for row in PLAN['schedule']:
            task = task_map[row['instance_id']]
            directory = RUN_DIR / 'runs' / f"{row['index']:02d}_{row['arm']}_{task.instance_id}"
            directory.mkdir(parents=True, exist_ok=False)
            arm_dir = ARMS_DIR / row['arm']
            cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots',
                results_dir=directory / 'harness', submission_dir=arm_dir, models=models,
                sandbox='subprocess', wheels_dir=DATA_DIR / 'wheels',
                graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
                timeout_seconds=profile['timeout_seconds'],
                verification_timeout_seconds=profile['verification_timeout_seconds'],
                max_time_minutes=profile['max_time_minutes'], max_tool_calls=profile['max_tool_calls'],
                limits=limits, generation_constraints=constraints, adapter_manifest=adapters,
                concurrency=1, enable_sandbox_testing=True,
                context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
                events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
                    token_threshold=14336, event_retention_size=5), verbose=False)
            compile_submission(arm_dir, SwegemmaContext().create_tools(), models,
                               limits=limits, generation_constraints=constraints)
            evaluator = Evaluator(cfg)
            context = SwegemmaContext(docker_manager=evaluator.docker, task=task,
                graph_dir=cfg.graph_dir, embeddings_dir=cfg.embeddings_dir,
                budget=cfg.budget, harness=cfg.harness)
            print('START', json.dumps(row), flush=True)
            start = time.monotonic()
            try:
                record = await evaluate_with_deadline(evaluator, task, context, directory,
                    profile['task_wall_minutes'] * 60, server_instance)
                raw = json.loads((directory / 'result.json').read_text())
                output = (directory / 'phase2.log').read_text()
                patch = (directory / 'agent.patch').read_text()
                record.update(classify(raw, output, patch))
                record.update(outer_timeout=False, execution_error=None,
                    agent_timeout=bool(re.search(r'time.?out|time budget', str(raw.get('error') or ''), re.I)))
            except Exception as exc:
                record = dict(instance_id=task.instance_id, resolved=False,
                    total_llm_calls=context.llm_calls_used, tool_calls=context.tool_calls_used,
                    duration_seconds=time.monotonic() - start, patch_application_valid=False,
                    phase2_invoked=False, collection_failed=False, outer_timeout=isinstance(exc, TimeoutError),
                    execution_error=type(exc).__name__ + ': ' + str(exc))
                exception_record(directory, 'comparison_task', exc)
                if isinstance(exc, TimeoutError):
                    # Synchronous cleanup can outlive cancellation. Stop rather than overlap runs.
                    failure = 'Outer timeout; remaining runs withheld to prevent cross-run interference'
            record.update(arm=row['arm'], category=row['category'], order=row['index'],
                          base_commit=task.base_commit, run_directory=str(directory))
            trace_path = directory / 'trace_legacy.json'
            if trace_path.exists():
                entries = json.loads(trace_path.read_text())['entries']
                record['trace_tool_calls'] = sum(e['type'] == 'tool_call' for e in entries)
                record['prompt_tokens'] = sum(e.get('usage', {}).get('prompt_tokens', 0) for e in entries)
                record['completion_tokens'] = sum(e.get('usage', {}).get('completion_tokens', 0) for e in entries)
            records.append(record)
            write_json(directory / 'comparison_record.json', record)
            write_json(RUN_DIR / 'comparison_records.json', records)
            write_json(RUN_DIR / 'comparison_summary.json', summarize(records, PLAN['schedule']))
            print('RESULT', json.dumps(record), flush=True)
            if failure:
                break
    except BaseException as exc:
        failure = type(exc).__name__ + ': ' + str(exc)
        exception_record(RUN_DIR, 'comparison', exc)
        raise
    finally:
        write_json(RUN_DIR / 'comparison_summary.json', summarize(records, PLAN['schedule']))
        write_json(RUN_DIR / 'comparison_status.json', dict(
            completed=len(records) == len(PLAN['schedule']), execution_failure=failure,
            observed_runs=len(records), planned_runs=len(PLAN['schedule']),
            unrun=[r for r in PLAN['schedule'] if r['index'] not in {x['order'] for x in records}],
            competition_score=None))
        preserve_server_log(server_instance, RUN_DIR)
        server_instance.stop()
        handler.flush()
        handler.close()
        print('EVIDENCE ZIP', bundle_run(RUN_DIR), flush=True)


## Frozen paired experiment input

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
SESSION_STARTED = time.monotonic()
RUN_DIR = WORKING_DIR / 'v1_pilot' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
server_instance = None
try:
    inputs = discover_inputs(INPUT_ROOT)
    ARMS_DIR = RUN_DIR / 'arms'
    PLAN = unpack_experiment(inputs['zip'], ARMS_DIR)
    profile = PLAN['profile']
    DATA_DIR, WHEELHOUSE_DIR, MODEL_PATH = inputs['data'], inputs['wheelhouse'], inputs['model']
    AGENT_DIR = ARMS_DIR / 'v0'
    TASK_INSTANCE_ID = None
    write_json(RUN_DIR / 'registered_plan.json', PLAN)
    write_json(RUN_DIR / 'manifest.json', dict(model=MODEL_NAME, inputs=inputs,
        payload_sha256=hashlib.sha256(inputs['zip'].read_bytes()).hexdigest(),
        experiment=PLAN['experiment'], profile=profile, competition_score=None))
    print('EVIDENCE DIRECTORY', RUN_DIR, flush=True)
except BaseException as exc:
    exception_record(RUN_DIR, 'comparison_inputs', exc)
    print('EVIDENCE ZIP', bundle_run(RUN_DIR), flush=True)
    raise


## Official offline installation

In [ ]:
import glob
import importlib
import os
import subprocess
import sys

# Same offline routing and dependency installation as official Notebook v3.
for name, value in {
    'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_FLAX': '1', 'TRANSFORMERS_OFFLINE': '1',
    'HF_HUB_OFFLINE': '1', 'VLLM_NO_USAGE_STATS': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1',
    'NCCL_DEBUG': 'WARN', 'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'LITELLM_TELEMETRY': 'False',
    'LITELLM_LOG': 'WARNING', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'OTEL_SDK_DISABLED': 'true', 'OTEL_PYTHON_DISABLED': 'true',
}.items():
    os.environ[name] = value
os.environ.setdefault('LOCAL_API_KEY', 'EMPTY')
os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')
try:
    gpu_query = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total',
                               '--format=csv,noheader'], check=True, capture_output=True, text=True, timeout=30)
    (RUN_DIR / 'gpu.txt').write_text(gpu_query.stdout)
    print(gpu_query.stdout)
    # Official Notebook workaround for broken cutlass .pth hooks.
    for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
        for path in glob.glob(pattern):
            Path(path).unlink(missing_ok=True)
    tmp_whl = RUN_DIR / 'wheelhouse_links'
    tmp_whl.mkdir()
    for wheel in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in wheel.name.lower():
            continue
        name = wheel.name.replace('cu128', '+cu128') if 'cu128' in wheel.name and '+' not in wheel.name else wheel.name
        (tmp_whl / name).symlink_to(wheel)
    wheels = sorted(str(p) for p in tmp_whl.glob('*.whl'))
    if not wheels:
        raise RuntimeError('Mounted official wheelhouse is empty')
    with (RUN_DIR / 'installation.log').open('w') as output:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                        '--force-reinstall', *wheels], stdout=output, stderr=subprocess.STDOUT, check=True, timeout=1800)
    importlib.invalidate_caches()
    # Link staging is not an artifact; only original file names are retained.
    shutil.rmtree(tmp_whl)
    from importlib.metadata import version
    packages = {name: version(name) for name in
                ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'google-genai',
                 'vllm', 'torch', 'transformers', 'litellm')}
    write_json(RUN_DIR / 'packages.json', packages)
    print('Installed official package versions:', packages)
    if any(packages[name] != expected for name, expected in {
        'swegemma': '0.2.11', 'adk-submission': '0.2.13', 'adk-eval-core': '0.1.0',
        'google-adk': '1.36.1', 'google-genai': '2.11.0'}.items()):
        raise RuntimeError('Official Harness version changed; review APIs before running')
except BaseException as exc:
    # Avoid copying wheel symlinks into a large diagnostic ZIP after install failure.
    if 'tmp_whl' in globals() and tmp_whl.exists():
        shutil.rmtree(tmp_whl)
    exception_record(RUN_DIR, 'gpu_or_installation', exc)
    print('Diagnostics ZIP:', bundle_run(RUN_DIR))
    raise


## Official task and model preflight

In [ ]:
try:
    import inspect
    import torch
    from adk_submission import (VllmConfig, VllmServer, compile_submission, discover_adapters,
                                validate_directory)
    from adk_submission.discovery import discover_declared_models
    from adk_submission.yaml_loader import load_yaml
    from swegemma.config import EvalConfig, build_submission_limits
    from swegemma.context import SwegemmaContext
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    from swegemma.deduplication import resolve_task_snapshot_paths
    
    assert torch.cuda.is_available() and torch.cuda.device_count() > 0, 'CUDA GPU required; no CPU fallback'
    gpu_count = torch.cuda.device_count()
    tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    assert discover_declared_models(AGENT_DIR) == {MODEL_NAME}, 'Only the required 31B model is allowed'
    adapters = discover_adapters(str(AGENT_DIR))
    assert not adapters.adapters, 'V0 must not contain LoRA adapters'
    raw_agent = load_yaml(AGENT_DIR / 'agent.yaml', AGENT_DIR)
    allowed_fields = {'agent_class', 'name', 'model', 'description', 'instruction', 'tools'}
    assert set(raw_agent) <= allowed_fields and raw_agent.get('agent_class') == 'LlmAgent'
    required_tools = {'get_status', 'run_command', 'read_file', 'edit_file', 'write_file', 'submit_patch'}
    assert set(raw_agent['tools']) == required_tools and len(raw_agent['tools']) == len(required_tools)
    assert required_tools <= SwegemmaContext().create_tools().keys()
    assert 'context' in inspect.signature(Evaluator.evaluate_task).parameters
    assert 'format' in inspect.signature(__import__('adk_eval_core.tracing', fromlist=['SessionTrace']).SessionTrace.save).parameters
    limits, constraints = build_submission_limits()
    validate_directory(AGENT_DIR, limits=limits)
    TASKS_PATH = DATA_DIR / 'tasks.jsonl'
    tasks = load_tasks(TASKS_PATH)
    assert tasks, 'No published tasks in the official input'
    task_by_id = {t.instance_id: t for t in tasks}
    selected = []
    for row in PLAN['schedule']:
        task = task_by_id[row['instance_id']]
        assert task.base_commit == row['base_commit'], 'Official task base changed'
        assert hashlib.sha256(task.problem_statement.encode()).hexdigest() == row['problem_sha256'], 'Task prompt changed'
        if task not in selected:
            selected.append(task)
    for task in selected:
        assert not task.repo.startswith('test/mock_repo'), 'Synthetic task cannot satisfy this run'
        snapshot, _, _ = resolve_task_snapshot_paths(DATA_DIR / 'snapshots', task.instance_id, task.repo)
        assert snapshot.is_file(), f'Missing official snapshot for {task.instance_id}'
    # Do not print gold patches or feed them to the model.
    write_json(RUN_DIR / 'selected_tasks.json', [dict(instance_id=t.instance_id, repo=t.repo,
        base_commit=t.base_commit, public_test_spec_present=bool(t.test_patch or t.FAIL_TO_PASS)) for t in selected])
    print('Selected public tasks:', [t.instance_id for t in selected])
    print('GPU count / tensor parallel:', gpu_count, tp_size)
    print('Profile:', profile)
except BaseException as exc:
    exception_record(RUN_DIR, 'harness_preflight', exc)
    print('Diagnostics ZIP:', bundle_run(RUN_DIR))
    raise


## Shared 31B server

In [ ]:
import urllib.request
from dataclasses import asdict

# Native Harness logs preserve ADK output/tool events while a task is running.
# No HTTP debug logging or credential/environment dumps.
handler = logging.FileHandler(RUN_DIR / 'harness.log', encoding='utf-8')
handler.setFormatter(logging.Formatter('%(asctime)s %(name)s %(levelname)s %(message)s'))
for name in ('swegemma', 'adk_submission'):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.addHandler(handler)
try:
    if not admission_seconds(SESSION_STARTED, profile['task_wall_minutes']):
        raise TimeoutError('Insufficient session time for model startup and a complete task slot')
    vllm_cfg = VllmConfig(model=str(MODEL_PATH), host='127.0.0.1', port=8000,
        tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
        gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=False,
        scheduler_reserve_full_isl=False, tensor_parallel_size=tp_size, startup_timeout=1200)
    write_json(RUN_DIR / 'vllm_config.json', asdict(vllm_cfg))
    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
    server_instance.start()
    preserve_server_log(server_instance, RUN_DIR)
    key = os.environ['LOCAL_API_KEY']
    request = urllib.request.Request(server_instance.base_url + '/models',
                                     headers={'Authorization': 'Bearer ' + key})
    with urllib.request.urlopen(request, timeout=30) as response:
        served = json.load(response)
    write_json(RUN_DIR / 'served_models.json', served)
    assert any(item.get('id') in (str(MODEL_PATH), MODEL_NAME) for item in served.get('data', [])), 'Unexpected served model'
    models = server_instance.create_model_registry(aliases=[MODEL_NAME], model_prefix='openai/', api_key=key)
    compiled = compile_submission(AGENT_DIR, SwegemmaContext().create_tools(), models,
                                 limits=limits, generation_constraints=constraints)
    assert not compiled.sub_agents
    write_json(RUN_DIR / 'run_status.json', {'stage': 'model_ready', 'completed': False,
               'real_task_completed': False, 'competition_score': None})
    print('Official server and compiler ready; no task inference has run yet.')
except BaseException as exc:
    exception_record(RUN_DIR, 'vllm_startup_or_compile', exc)
    # VllmServer.start() deletes its temporary log on failure; output contains its captured log.
    if getattr(exc, 'output', None):
        (RUN_DIR / 'vllm_startup_failure.log').write_text(str(exc.output))
    if server_instance is not None:
        preserve_server_log(server_instance, RUN_DIR)
        server_instance.stop()
    handler.flush()
    print('Diagnostics ZIP:', bundle_run(RUN_DIR))
    raise


## Sequential paired runs, fresh task contexts

In [ ]:
await run_comparison()
